# BiLSTM multietiqueta — Seguridad y subcategorías


Este experimento usa exclusivamente la tarea multietiqueta de Seguridad: `Security` y sus seis
subcategorías. El protocolo de carga, validación, partición y métricas vive en
`security_multilabel_common.py`, por lo que los cuatro modelos consumen exactamente el mismo
manifiesto `artifacts/security_split_manifest_seed42.csv`.

El CSV actual contiene preanotaciones automáticas. `ALLOW_PROVISIONAL_LABELS=True` permite este
prototipo; cambiarlo a `False` exige `expert_validated` o `consensus` en todos los positivos.
Test está reservado y no interviene en entrenamiento, selección de configuración ni umbral.


## 1. Configuración y reproducibilidad


In [ ]:
from __future__ import annotations

import json
import random
import re
import shutil
from collections import Counter
from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from torch.utils.data import DataLoader, Dataset

from security_multilabel_common import (
    DATA_PATH, LABEL_COLS, SEED, SPLIT_PATH, TEXT_COL,
    get_or_create_splits, is_better, load_security_dataset,
    multilabel_metrics, prediction_frame, save_json, seed_everything, select_per_label_thresholds,
)

ALLOW_PROVISIONAL_LABELS = True
USE_TRAIN_AUGMENTATION = False  # activar solo después de revisión experta
USE_MODERATE_POS_WEIGHT = False  # comparar como ablación tras corregir etiquetas
RUN_HPARAM_SEARCH = False
RUN_FINAL_TEST = False

seed_everything(SEED)
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
ARTIFACTS_DIR = Path("artifacts")
ARTIFACTS_DIR.mkdir(parents=True, exist_ok=True)
print("Device:", DEVICE, "| PyTorch:", torch.__version__)


In [ ]:
BILSTM_CONFIG = {
    "embedding_dim": 100, "hidden_units": 64, "num_layers": 1, "batch_size": 32,
    "epochs": 30, "dropout": 0.2, "learning_rate": 0.001,
    "optimizer": "Adam", "threshold": 0.50,
}
INITIAL_CONFIG = BILSTM_CONFIG.copy()
SEARCH_SPACE = {
    "embedding_dim": [64, 100, 128], "hidden_units": [32, 64, 128], "num_layers": [1, 2],
    "batch_size": [16, 32, 50], "epochs": [20, 30, 50], "dropout": [0.2, 0.3, 0.5],
    "learning_rate": [0.0005, 0.001, 0.002], "optimizer": ["Adam"], "threshold": [0.40, 0.60],
}
SEARCH_TRIALS = [
    INITIAL_CONFIG.copy(),
    {**INITIAL_CONFIG, "embedding_dim": 64, "hidden_units": 32, "num_layers": 2,
     "batch_size": 16, "epochs": 20, "dropout": 0.3, "learning_rate": 0.0005, "threshold": 0.45},
    {**INITIAL_CONFIG, "embedding_dim": 128, "hidden_units": 128, "batch_size": 50,
     "epochs": 50, "dropout": 0.5, "learning_rate": 0.002, "threshold": 0.55},
]
MODEL_KEY = "bilstm_security"
CHECKPOINT_DIR = Path("checkpoints/bilstm_security")
GRAD_CLIP = 1.0


## 2. Dataset validado y partición común


In [ ]:
df = load_security_dataset(DATA_PATH, allow_provisional_labels=ALLOW_PROVISIONAL_LABELS)
train_df, val_df, test_df, split_manifest = get_or_create_splits(df, SPLIT_PATH)

print("Etiquetas (orden fijo):", LABEL_COLS)
print("Particiones:", {"train": len(train_df), "val": len(val_df), "test": len(test_df)})
print("Método de estratificación:", split_manifest["stratification_method"].iloc[0])
if "fallback" in split_manifest["stratification_method"].iloc[0]:
    print("AVISO: no está instalado iterative-stratification; se usó el fallback determinista, menos fiel.")
for name, part in (("train", train_df), ("val", val_df), ("test", test_df)):
    print(name, part[LABEL_COLS].mean().round(3).to_dict())
assert not split_manifest.query("split in ['val', 'test']")["is_synthetic"].astype(bool).any()


# El aumento queda pausado hasta completar la revisión experta de etiquetas raras.
from security_train_augmentation import augment_balanced_subcategories
real_train_df = train_df.copy()
if USE_TRAIN_AUGMENTATION:
    train_df = augment_balanced_subcategories(real_train_df)
else:
    train_df = real_train_df
print("Train real / utilizado:", len(real_train_df), "/", len(train_df))
print("Positivos utilizados:", train_df[LABEL_COLS].sum().astype(int).to_dict())

# Peso moderado opcional: sqrt(N_neg/N_pos), limitado a 10 y calculado solo con train.
positive_counts = train_df[LABEL_COLS].sum().to_numpy(dtype=float)
negative_counts = len(train_df) - positive_counts
moderate_pos_weight = np.minimum(np.sqrt(negative_counts / np.maximum(positive_counts, 1.0)), 10.0)
POS_WEIGHT = torch.tensor(moderate_pos_weight, dtype=torch.float32, device=DEVICE)
print("Moderate pos_weight:", dict(zip(LABEL_COLS, moderate_pos_weight.round(3))))

def make_criterion():
    return nn.BCEWithLogitsLoss(pos_weight=POS_WEIGHT if USE_MODERATE_POS_WEIGHT else None)


## 3. Tokenización y vocabulario construido solo con train


In [ ]:
TOKEN_PATTERN = re.compile(r"[A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+(?:['’-][A-Za-zÁÉÍÓÚÜÑáéíóúüñ0-9]+)?")
PAD_TOKEN, UNK_TOKEN = "<PAD>", "<UNK>"
PAD_IDX, UNK_IDX = 0, 1
MIN_FREQ, MAX_LEN_CAP, MAX_LEN_PERCENTILE = 1, 256, 95

def tokenize(text):
    return TOKEN_PATTERN.findall(str(text).lower())

def build_vocab(texts, min_freq=MIN_FREQ):
    counts = Counter(token for text in texts for token in tokenize(text))
    vocab = {PAD_TOKEN: PAD_IDX, UNK_TOKEN: UNK_IDX}
    for token, frequency in counts.most_common():
        if frequency >= min_freq:
            vocab.setdefault(token, len(vocab))
    return vocab

def encode_text(text, vocab, max_len):
    ids = [vocab.get(token, UNK_IDX) for token in tokenize(text)][:max_len] or [UNK_IDX]
    length = len(ids)
    return ids + [PAD_IDX] * (max_len - length), length

vocab = build_vocab(train_df[TEXT_COL])  # solo train
train_lengths = np.array([max(1, len(tokenize(text))) for text in train_df[TEXT_COL]])
MAX_LEN = max(5, min(MAX_LEN_CAP, int(np.ceil(np.percentile(train_lengths, MAX_LEN_PERCENTILE)))))
print("Vocabulario:", len(vocab), "| MAX_LEN:", MAX_LEN)

class RequirementsDataset(Dataset):
    def __init__(self, frame):
        self.frame = frame.reset_index(drop=True)
    def __len__(self):
        return len(self.frame)
    def __getitem__(self, index):
        row = self.frame.iloc[index]
        input_ids, length = encode_text(row[TEXT_COL], vocab, MAX_LEN)
        return {
            "id": str(row["ID"]),
            "input_ids": torch.tensor(input_ids, dtype=torch.long),
            "length": torch.tensor(length, dtype=torch.long),
            "labels": torch.tensor(row[LABEL_COLS].to_numpy(dtype=np.float32)),
        }

def make_loaders(batch_size):
    generator = torch.Generator().manual_seed(SEED)
    return (
        DataLoader(RequirementsDataset(train_df), batch_size=batch_size, shuffle=True, generator=generator),
        DataLoader(RequirementsDataset(val_df), batch_size=batch_size, shuffle=False),
        DataLoader(RequirementsDataset(test_df), batch_size=batch_size, shuffle=False),
    )


## 4. Arquitectura


In [ ]:
class BiLSTMClassifier(nn.Module):
    def __init__(self, config):
        super().__init__()
        self.embedding = nn.Embedding(len(vocab), config["embedding_dim"], padding_idx=PAD_IDX)
        self.lstm = nn.LSTM(config["embedding_dim"], config["hidden_units"],
            num_layers=config["num_layers"], batch_first=True, bidirectional=True,
            dropout=config["dropout"] if config["num_layers"] > 1 else 0.0)
        self.dropout = nn.Dropout(config["dropout"])
        self.classifier = nn.Linear(2 * config["hidden_units"], len(LABEL_COLS))
    def forward(self, input_ids, lengths):
        packed = nn.utils.rnn.pack_padded_sequence(
            self.embedding(input_ids), lengths.cpu(), batch_first=True, enforce_sorted=False)
        _, (hidden, _) = self.lstm(packed)
        features = torch.cat((hidden[-2], hidden[-1]), dim=1)
        return self.classifier(self.dropout(features))  # 7 logits independientes

def build_model(config):
    return BiLSTMClassifier(config)


In [ ]:
CHECKPOINT_DIR.mkdir(parents=True, exist_ok=True)
BEST_CHECKPOINT, LAST_CHECKPOINT = CHECKPOINT_DIR / "best.pt", CHECKPOINT_DIR / "last.pt"
HISTORY_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_history.csv"
TRIALS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_trials.csv"
VAL_PRED_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_validation_predictions.csv"
VAL_LABEL_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_validation_metrics_by_label.csv"
WINNER_CONFIG_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_winning_config.json"
CURVES_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_curves.png"
TEST_PRED_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_predictions.csv"
TEST_LABEL_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_metrics_by_label.csv"
TEST_METRICS_PATH = ARTIFACTS_DIR / f"{MODEL_KEY}_test_metrics.json"


## 5. BCE, entrenamiento y selección por macro-F1 / recall de Security


In [ ]:
def train_one_epoch(model, loader, criterion, optimizer, grad_clip=None):
    model.train()
    total = 0.0
    for batch in loader:
        x, lengths = batch["input_ids"].to(DEVICE), batch["length"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)
        optimizer.zero_grad(set_to_none=True)
        loss = criterion(model(x, lengths), labels)
        loss.backward()
        if grad_clip is not None:
            nn.utils.clip_grad_norm_(model.parameters(), grad_clip)
        optimizer.step()
        total += loss.item() * len(labels)
    return total / len(loader.dataset)

def evaluate(model, loader, criterion, threshold):
    model.eval()
    ids, truths, probabilities, total = [], [], [], 0.0
    with torch.no_grad():
        for batch in loader:
            x, lengths = batch["input_ids"].to(DEVICE), batch["length"].to(DEVICE)
            labels = batch["labels"].to(DEVICE)
            logits = model(x, lengths)
            total += criterion(logits, labels).item() * len(labels)
            ids.extend(batch["id"])
            truths.append(labels.cpu().numpy())
            probabilities.append(torch.sigmoid(logits).cpu().numpy())
    metrics = multilabel_metrics(np.vstack(truths), np.vstack(probabilities), threshold)
    metrics.update({"loss": total / len(loader.dataset), "ids": ids,
                    "y_true": np.vstack(truths), "y_prob": np.vstack(probabilities)})
    return metrics

def checkpoint_payload(model, optimizer, epoch, metrics, config):
    return {
        "epoch": epoch, "model_state_dict": model.state_dict(),
        "optimizer_state_dict": optimizer.state_dict(), "config": config,
        "val_macro_f1": metrics["macro_f1"], "val_security_recall": metrics["security_recall"],
        "val_loss": metrics["loss"], "vocab": vocab, "max_len": MAX_LEN,
        "label_cols": LABEL_COLS, "threshold": config["threshold"], "seed": SEED,
    }

def run_configuration(config, trial_id):
    seed_everything(SEED + trial_id)
    train_loader, val_loader, _ = make_loaders(config["batch_size"])
    model = build_model(config).to(DEVICE)
    criterion = make_criterion()
    optimizer = torch.optim.Adam(model.parameters(), lr=config["learning_rate"])
    trial_dir = CHECKPOINT_DIR / f"trial_{trial_id:03d}"
    trial_dir.mkdir(parents=True, exist_ok=True)
    history, best = [], None
    for epoch in range(1, config["epochs"] + 1):
        train_loss = train_one_epoch(model, train_loader, criterion, optimizer, GRAD_CLIP)
        current = evaluate(model, val_loader, criterion, config["threshold"])
        row = {"trial_id": trial_id, "epoch": epoch, "train_loss": train_loss,
               "val_loss": current["loss"], "val_macro_f1": current["macro_f1"],
               "val_security_recall": current["security_recall"], "val_micro_f1": current["micro_f1"]}
        history.append(row)
        torch.save(checkpoint_payload(model, optimizer, epoch, current, config), trial_dir / "last.pt")
        if is_better(current, best):
            best = {"macro_f1": current["macro_f1"], "security_recall": current["security_recall"],
                    "loss": current["loss"], "epoch": epoch}
            torch.save(checkpoint_payload(model, optimizer, epoch, current, config), trial_dir / "best.pt")
        print(f"trial={trial_id} epoch={epoch}/{config['epochs']} val_macro_f1={current['macro_f1']:.4f} "
              f"security_recall={current['security_recall']:.4f}")
    return best, history, trial_dir


## 6. Trials, checkpoint ganador y validación


In [ ]:
configs = SEARCH_TRIALS if RUN_HPARAM_SEARCH else [INITIAL_CONFIG.copy()]
trial_rows, all_history, trial_dirs = [], [], {}
for trial_id, config in enumerate(configs):
    best, history, trial_dir = run_configuration(config.copy(), trial_id)
    trial_dirs[trial_id] = trial_dir
    all_history.extend(history)
    trial_rows.append({"trial_id": trial_id, **config, "val_macro_f1": best["macro_f1"],
                       "val_security_recall": best["security_recall"], "val_loss": best["loss"],
                       "best_epoch": best["epoch"]})

trials_df = pd.DataFrame(trial_rows).sort_values(
    ["val_macro_f1", "val_security_recall"], ascending=[False, False], kind="stable"
).reset_index(drop=True)
trials_df.to_csv(TRIALS_PATH, index=False, encoding="utf-8-sig")
history_df = pd.DataFrame(all_history)
history_df.to_csv(HISTORY_PATH, index=False, encoding="utf-8-sig")
winner = trials_df.iloc[0].to_dict()
winner_id = int(winner["trial_id"])
shutil.copy2(trial_dirs[winner_id] / "best.pt", BEST_CHECKPOINT)
shutil.copy2(trial_dirs[winner_id] / "last.pt", LAST_CHECKPOINT)

checkpoint = torch.load(BEST_CHECKPOINT, map_location=DEVICE, weights_only=False)
winning_config = checkpoint["config"]
model = build_model(winning_config).to(DEVICE)
model.load_state_dict(checkpoint["model_state_dict"])
criterion = make_criterion()
_, val_loader, test_loader = make_loaders(winning_config["batch_size"])
val_metrics = evaluate(model, val_loader, criterion, winning_config["threshold"])

# Ajustar un umbral por etiqueta usando solo validation (test continúa reservado).
SELECTED_THRESHOLDS, threshold_df = select_per_label_thresholds(
    val_metrics["y_true"], val_metrics["y_prob"]
)
winning_config = {
    **winning_config,
    "thresholds_by_label": dict(zip(LABEL_COLS, SELECTED_THRESHOLDS.tolist())),
}
selected = multilabel_metrics(val_metrics["y_true"], val_metrics["y_prob"], SELECTED_THRESHOLDS)
selected.update({key: val_metrics[key] for key in ["loss", "ids", "y_true", "y_prob"]})
val_metrics = selected
threshold_df.to_csv(
    ARTIFACTS_DIR / f"{MODEL_KEY}_threshold_search.csv", index=False, encoding="utf-8-sig"
)
print("Umbrales seleccionados solo con validation:", winning_config["thresholds_by_label"])
checkpoint["config"] = winning_config
checkpoint["thresholds_by_label"] = winning_config["thresholds_by_label"]
torch.save(checkpoint, BEST_CHECKPOINT)


prediction_frame(val_metrics["ids"], val_metrics).to_csv(VAL_PRED_PATH, index=False, encoding="utf-8-sig")
val_metrics["per_label"].to_csv(VAL_LABEL_METRICS_PATH, index=False, encoding="utf-8-sig")
save_json(WINNER_CONFIG_PATH, {"config": winning_config, "best_epoch": checkpoint["epoch"],
                               "val_macro_f1": val_metrics["macro_f1"],
                               "val_security_recall": val_metrics["security_recall"]})
display(trials_df)
display(val_metrics["per_label"])
print({key: val_metrics[key] for key in ["macro_f1", "security_recall", "micro_f1", "subset_accuracy", "hamming_loss"]})


## 7. Curvas


In [ ]:
winner_history = history_df[history_df["trial_id"].eq(winner_id)]
fig, axes = plt.subplots(1, 2, figsize=(12, 4))
axes[0].plot(winner_history["epoch"], winner_history["train_loss"], label="train_loss")
axes[0].plot(winner_history["epoch"], winner_history["val_loss"], label="val_loss")
axes[1].plot(winner_history["epoch"], winner_history["val_macro_f1"], label="val_macro_f1")
axes[1].plot(winner_history["epoch"], winner_history["val_security_recall"], label="val_security_recall")
for axis in axes:
    axis.grid(alpha=.3); axis.legend(); axis.set_xlabel("Época")
fig.tight_layout()
fig.savefig(CURVES_PATH, dpi=150, bbox_inches="tight")
plt.show()


## 8. Inferencia y test reservado


In [ ]:
def predict_text(text):
    ids, length = encode_text(text, vocab, MAX_LEN)
    x = torch.tensor([ids], dtype=torch.long, device=DEVICE)
    lengths = torch.tensor([length], dtype=torch.long, device=DEVICE)
    model.eval()
    with torch.no_grad():
        probabilities = torch.sigmoid(model(x, lengths)).squeeze(0).cpu().numpy()
    thresholds = np.array([winning_config.get("thresholds_by_label", {}).get(label, winning_config["threshold"]) for label in LABEL_COLS])
    return pd.DataFrame({"label": LABEL_COLS, "probability": probabilities,
                         "predicted": (probabilities >= thresholds).astype(int), "threshold": thresholds})

display(predict_text("El sistema deberá autenticar al usuario antes de permitir el acceso."))

if RUN_FINAL_TEST:
    # Evaluar una sola vez, después de fijar configuración y umbral con validation.
    test_metrics = evaluate(model, test_loader, criterion, SELECTED_THRESHOLDS)
    prediction_frame(test_metrics["ids"], test_metrics).to_csv(TEST_PRED_PATH, index=False, encoding="utf-8-sig")
    test_metrics["per_label"].to_csv(TEST_LABEL_METRICS_PATH, index=False, encoding="utf-8-sig")
    save_json(TEST_METRICS_PATH, {key: test_metrics[key] for key in
              ["macro_f1", "security_recall", "micro_f1", "subset_accuracy", "hamming_loss", "loss", "threshold"]})
    display(test_metrics["per_label"])
else:
    print("Test reservado: RUN_FINAL_TEST = False")
